In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [102]:
spark = SparkSession.builder \
    .appName("skincare_database") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .getOrCreate() 

df = spark.read.csv(
    'Global skincare and Beauty e-store_E-commerce Analysis.csv', 
    header=True, 
    inferSchema=True
)

df.toPandas().head()

,Row ID,Order ID,Order Date,Customer ID,Segment,City,State,Country,Country latitude,Country longitude,Region,Market,Subcategory,Category,Product,Quantity,Sales,Discount,Profit
0,46682,IZ-2012-LW699061-40911,03/01/2020,LW-699061,Corporate,Mosul,Ninawa,Iraq,33.223191,43.679291,Western Asia,Asia Pacific,"bath oils, bubbles and soaks",Body care,Head & Shoulders Classic Clean Shampoo,20,600,0.0,300.0
1,10124,US-2012-BT1130518-40912,04/01/2020,BT-1130518,Self-Employed,Pilar,Alagoas,Brazil,-14.235004,-51.925280,South America,LATAM,"bath oils, bubbles and soaks",Body care,Kiehl's Ultra Facial Overnight Hydrating Masque,2,40,0.6,-4.0
2,9067,MX-2012-AW1093031-40912,04/01/2020,AW-1093031,Self-Employed,Santiago de Cuba,Santiago de Cuba,Cuba,21.521757,-77.781167,Caribbean,LATAM,"bath oils, bubbles and soaks",Body care,Golden Vine Bracelet,1,151,0.0,75.5
3,130,MX-2012-BT1130531-40912,04/01/2020,BT-1130531,Self-Employed,Manzanillo,Granma,Cuba,21.521757,-77.781167,Caribbean,LATAM,"bath oils, bubbles and soaks",Body care,Kiehl's Crème de Corps Smoothing Oil-to-Foam B...,4,40,0.0,20.0
4,24072,IN-2012-KM1666027-40914,06/01/2020,KM-1666027,Consumer,Huadian,Jilin,China,35.861660,104.195397,Eastern Asia,Asia Pacific,"bath oils, bubbles and soaks",Body care,NARS Single Eyeshadow Sophia Cool Brown,1,11,0.0,5.5


In [103]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Country latitude: double (nullable = true)
 |-- Country longitude: double (nullable = true)
 |-- Region: string (nullable = true)
 |-- Market: string (nullable = true)
 |-- Subcategory: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Sales: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [104]:
# Check Duplicate
df.groupBy(df.columns).count().filter("count > 1").show()

+------+--------+----------+-----------+-------+----+-----+-------+----------------+-----------------+------+------+-----------+--------+-------+--------+-----+--------+------+-----+
|Row ID|Order ID|Order Date|Customer ID|Segment|City|State|Country|Country latitude|Country longitude|Region|Market|Subcategory|Category|Product|Quantity|Sales|Discount|Profit|count|
+------+--------+----------+-----------+-------+----+-----+-------+----------------+-----------------+------+------+-----------+--------+-------+--------+-----+--------+------+-----+
+------+--------+----------+-----------+-------+----+-----+-------+----------------+-----------------+------+------+-----------+--------+-------+--------+-----+--------+------+-----+



In [105]:
# Check Missing Values
df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).toPandas().head()

,Row ID,Order ID,Order Date,Customer ID,Segment,City,State,Country,Country latitude,Country longitude,Region,Market,Subcategory,Category,Product,Quantity,Sales,Discount,Profit
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [106]:
# Rename All Columns
new_columns = [col.replace(" ", "_").lower() for col in df.columns]
df = df.toDF(*new_columns)

In [107]:
# Hapus Kolom Row ID
df = df.drop('row_id')

# Ubah tipe data order date menjadi date format
df = df.withColumn(
    "order_date",
    F.to_date("order_date", "dd/MM/yyyy")
)

In [108]:
# Membuat kolom harga per produk / retail_price
df = df.withColumn('retail_price', F.col('sales') / F.col('quantity'))
df.select('sales', 'quantity', 'retail_price').show(5)

+-----+--------+------------+
|sales|quantity|retail_price|
+-----+--------+------------+
|  600|      20|        30.0|
|   40|       2|        20.0|
|  151|       1|       151.0|
|   40|       4|        10.0|
|   11|       1|        11.0|
+-----+--------+------------+
only showing top 5 rows


In [109]:
dim_customer = df.select('customer_id', 'segment').dropDuplicates(['customer_id'])
dim_customer.show()

+------------+--------+
| customer_id| segment|
+------------+--------+
| AA-10315102|Consumer|
| AA-10315120|Consumer|
| AA-10315139|Consumer|
|AA-103151402|Consumer|
|AA-103151404|Consumer|
|AA-103151406|Consumer|
|  AA-1031545|Consumer|
|  AA-1031548|Consumer|
|  AA-1031558|Consumer|
|   AA-103157|Consumer|
|  AA-1031582|Consumer|
|   AA-103751|Consumer|
| AA-10375101|Consumer|
|AA-103751402|Consumer|
|AA-103751404|Consumer|
|AA-103751406|Consumer|
|AA-103751408|Consumer|
|  AA-1037545|Consumer|
|   AA-103755|Consumer|
|  AA-1037554|Consumer|
+------------+--------+
only showing top 20 rows


In [110]:
# Membuat dim_product
dim_product = df.select('product', 'subcategory', 'category', 'retail_price').dropDuplicates(['product'])
dim_product = dim_product.withColumn('product_id', (F.monotonically_increasing_id()+1))
cols = ["product_id"] + [c for c in dim_product.columns if c != "product_id"]
dim_product = dim_product.select(*cols)

dim_product.show(5, truncate=False)

+----------+------------------------------------------------------------------+----------------------------+---------+------------+
|product_id|product                                                           |subcategory                 |category |retail_price|
+----------+------------------------------------------------------------------+----------------------------+---------+------------+
|1         |Acqua di Parma Blu Mediterraneo Arancia di Capri Bath & Shower Gel|bath oils, bubbles and soaks|Body care|5.0         |
|2         |Aesop Geranium Leaf Hydrating Body Treatment                      |bath oils, bubbles and soaks|Body care|8.0         |
|3         |Aesop Petitgrain Reviving Body Gel                                |bath oils, bubbles and soaks|Body care|7.0         |
|4         |Aesop Rind Concentrate Body Balm                                  |bath oils, bubbles and soaks|Body care|7.0         |
|5         |Ahava Dead Sea Mineral Bath Salts                               

In [111]:
# Membuat dim_location
dim_location = df.select('city', 'state', 'country', 'region', 'market').dropDuplicates()
dim_location = dim_location.withColumn('location_id', (F.monotonically_increasing_id()+1))
cols = ["location_id"] + [c for c in dim_location.columns if c != "location_id"]
dim_location = dim_location.select(*cols)

dim_location.show(5, truncate=False)

+-----------+---------+----------------------+--------------+---------------+------------+
|location_id|city     |state                 |country       |region         |market      |
+-----------+---------+----------------------+--------------+---------------+------------+
|1          |Baruta   |Miranda               |Venezuela     |South America  |LATAM       |
|2          |Preston  |England               |United Kingdom|Northern Europe|Europe      |
|3          |Dalian   |Liaoning              |China         |Eastern Asia   |Asia Pacific|
|4          |Halle    |North Rhine-Westphalia|Germany       |Western Europe |Europe      |
|5          |Sonsonate|Sonsonate             |El Salvador   |Central America|LATAM       |
+-----------+---------+----------------------+--------------+---------------+------------+
only showing top 5 rows


In [112]:
# Membuat dim_time
dim_time = df.select('order_date').dropDuplicates()

dim_time = dim_time.withColumn("day", F.dayofmonth("order_date")) \
                   .withColumn("month", F.month("order_date")) \
                   .withColumn("year", F.year("order_date")) \
                   .withColumn("start_of_month", F.trunc("order_date", "month")) \
                   .withColumn("day_of_week", F.dayofweek("order_date")) # 1=Minggu, 7=Sabtu 

dim_time = dim_time.withColumn("is_weekend", 
                               F.when(F.col("day_of_week").isin(1, 7), 'Weekend').otherwise('Weekday'))

dim_time.show()

+----------+---+-----+----+--------------+-----------+----------+
|order_date|day|month|year|start_of_month|day_of_week|is_weekend|
+----------+---+-----+----+--------------+-----------+----------+
|2020-08-24| 24|    8|2020|    2020-08-01|          2|   Weekday|
|2021-06-22| 22|    6|2021|    2021-06-01|          3|   Weekday|
|2021-08-27| 27|    8|2021|    2021-08-01|          6|   Weekday|
|2021-10-11| 11|   10|2021|    2021-10-01|          2|   Weekday|
|2021-11-13| 13|   11|2021|    2021-11-01|          7|   Weekend|
|2021-12-18| 18|   12|2021|    2021-12-01|          7|   Weekend|
|2022-03-28| 28|    3|2022|    2022-03-01|          2|   Weekday|
|2022-07-31| 31|    7|2022|    2022-07-01|          1|   Weekend|
|2023-06-22| 22|    6|2023|    2023-06-01|          5|   Weekday|
|2023-07-15| 15|    7|2023|    2023-07-01|          7|   Weekend|
|2021-01-27| 27|    1|2021|    2021-01-01|          4|   Weekday|
|2020-01-21| 21|    1|2020|    2020-01-01|          3|   Weekday|
|2020-07-2

In [113]:
# Membuat fact_sales
fact_table = df.join(dim_customer, 'customer_id', 'left')\
               .join(dim_product, 'product', 'left')\
               .join(dim_location, ['city', 'state', 'country'], 'left')\
               .join(dim_time, 'order_date', 'left')

fact_sales = fact_table.select('order_id', 'order_date', 'customer_id', 'product_id', 'location_id', 'quantity', 'sales', 'discount', 'profit')
fact_sales.show()

+--------------------+----------+------------+----------+-----------+--------+-----+--------+------+
|            order_id|order_date| customer_id|product_id|location_id|quantity|sales|discount|profit|
+--------------------+----------+------------+----------+-----------+--------+-----+--------+------+
|IZ-2012-LW699061-...|2020-01-03|   LW-699061|      3101|       1265|      20|  600|     0.0| 300.0|
|US-2012-BT1130518...|2020-01-04|  BT-1130518|      1877|       2612|       2|   40|     0.6|  -4.0|
|MX-2012-AW1093031...|2020-01-04|  AW-1093031|       273|         18|       1|  151|     0.0|  75.5|
|MX-2012-BT1130531...|2020-01-04|  BT-1130531|      2041|       2302|       4|   40|     0.0|  20.0|
|IN-2012-KM1666027...|2020-01-06|  KM-1666027|      2478|       2164|       1|   11|     0.0|   5.5|
|HU-2012-ER385557-...|2020-01-10|   ER-385557|      2727|       2365|       8|  352|     0.0| 176.0|
|ID-2012-TN2104092...|2020-01-11|  TN-2104092|       305|       1018|       2|   36|     0.

In [116]:
dim_customer.coalesce(1).write.csv("dim_customer", header=True, mode="overwrite")
dim_product.coalesce(1).write.csv("dim_product", header=True, mode="overwrite")
dim_location.coalesce(1).write.csv("dim_location", header=True, mode="overwrite")
dim_time.coalesce(1).write.csv("dim_time", header=True, mode="overwrite")
fact_sales.coalesce(1).write.csv("fact_sales", header=True, mode="overwrite")